In [ ]:
!pip install -q gradio
import numpy as np, tensorflow as tf, tensorflow_datasets as tfds, gradio as gr
from PIL import Image, ImageOps

EPOCHS = 5
chars = [str(i) for i in range(10)] + [chr(65 + i) for i in range(26)]

# Load data (digits + capital letters)
(xd, yd), _ = tf.keras.datasets.mnist.load_data()
ds = tfds.load("emnist/letters", split="train", as_supervised=True)
xl, yl = zip(*[(i.squeeze().T, l + 9) for i, l in tfds.as_numpy(ds)])
x = np.concatenate([xd, np.array(xl)]).astype("float32")[..., None] / 255
y = np.concatenate([yd, np.array(yl)])

# CNN model
model = tf.keras.Sequential([
    tf.keras.layers.Input((28, 28, 1)),
    tf.keras.layers.Conv2D(32, 3, activation="relu"), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(36, activation="softmax")])
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])

# Training runs every time, so Epoch 1/5 ... 5/5 always shows
model.fit(x, y, epochs=EPOCHS, batch_size=128, validation_split=0.1, shuffle=True)

# App
def predict(img):
    if isinstance(img, dict): img = img["composite"]
    if img is None: return None
    bg = Image.new("RGBA", img.size, "white")
    img = ImageOps.invert(Image.alpha_composite(bg, img.convert("RGBA")).convert("L"))
    box = img.getbbox()
    if not box: return None
    img = img.crop(box); img.thumbnail((20, 20))
    c = Image.new("L", (28, 28)); c.paste(img, ((28 - img.width) // 2, (28 - img.height) // 2))
    p = model.predict(np.array(c, "float32")[None, ..., None] / 255, verbose=0)[0]
    return {chars[i]: float(p[i]) for i in np.argsort(p)[::-1][:5]}

gr.Interface(predict, gr.Sketchpad(image_mode="RGBA", type="pil"), gr.Label(num_top_classes=5),
             title="Digit & Letter Recognizer").launch(share=True)

Epoch 1/5
 432/1047 ━━━━━━━━━━━━━━━━━━━━ 1:20 132ms/step - accuracy: 0.5386 - loss: 1.6865